In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import sys
import subprocess
import torch
import os
import shutil
import logging
import pyNN.spiNNaker as sim

logging.getLogger().setLevel(logging.ERROR)
for logger_name in list(logging.root.manager.loggerDict.keys()):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

DATASET = "dvs_gesture"  # Choix : "nmnist", "cifar10", "dvs_gesture" "nepic_kitchen"
WEIGHTS_PATHS = {
    "nmnist": "networks/nmnist_best.pth",
    "cifar10": "networks/cifar10_best.pth",
    "dvs_gesture": "networks/dvs-gesture_best.pth"
}
PTH_FILE = WEIGHTS_PATHS.get(DATASET, "")
SIMULATION_TIME = 20.0

def format_weights(weight_matrix):
    connector_list = []
    for i in range(weight_matrix.shape[1]):
        for j in range(weight_matrix.shape[0]):
            connector_list.append((i, j, float(weight_matrix[j, i]), 1.0))
    return connector_list

def run_nmnist(pth_path, simulation_time):
    print("--- Deployment N-MNIST (SpikingMLP) on SpiNNaker ---")
    sim.setup(timestep=1.0)
    sim.set_number_of_neurons_per_core(sim.IF_curr_exp, 64)
    
    pop_in = sim.Population(2312, sim.SpikeSourcePoisson(rate=50.0), label="Input")
    pop_hidden = sim.Population(256, sim.IF_curr_exp(), label="Hidden")
    pop_out = sim.Population(10, sim.IF_curr_exp(), label="Output")
    
    if os.path.exists(pth_path):
        print(f"Load of the weights from : {pth_path}")
        state_dict = torch.load(pth_path, map_location="cpu", weights_only=True)
        if "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
            
        weights_hidden = state_dict["network.1.weight"].numpy()
        weights_out = state_dict["network.4.weight"].numpy()
        
        sim.Projection(pop_in, pop_hidden, sim.FromListConnector(format_weights(weights_hidden)))
        sim.Projection(pop_hidden, pop_out, sim.FromListConnector(format_weights(weights_out)))
    else:
        print("Weights file not found; using random weights for energy profiling.")
        sim.Projection(pop_in, pop_hidden, sim.FixedProbabilityConnector(0.1), sim.StaticSynapse(weight=0.5))
        sim.Projection(pop_hidden, pop_out, sim.FixedProbabilityConnector(0.5), sim.StaticSynapse(weight=0.5))

    pop_out.record(["spikes"])
    
    print(f"Hardware simulation in progress: ({simulation_time} ms)...")
    sim.run(simulation_time)
    spikes = pop_out.get_data("spikes")
    sim.end()
    
    print("Simulation completed with success.")
    return spikes

def run_vgg_topology(dataset_name, layer_sizes, simulation_time):
    print(f"--- Hardware Profiling {dataset_name} on SpiNNaker ---")
    print(f"Allocation of {sum(layer_sizes):,} neurons on ARM cores...")
    sim.setup(timestep=1.0)
    sim.set_number_of_neurons_per_core(sim.IF_curr_exp, 64)
    
    populations = []
    populations.append(sim.Population(layer_sizes[0], sim.SpikeSourcePoisson(rate=20.0), label="Input"))
    
    for i, size in enumerate(layer_sizes[1:]):
        pop = sim.Population(size, sim.IF_curr_exp(), label=f"Layer_{i+1}")
        populations.append(pop)
        
        prob = min(0.05, 1000.0 / layer_sizes[i])
        sim.Projection(
            populations[i], pop, 
            sim.FixedProbabilityConnector(p_connect=prob),
            sim.StaticSynapse(weight=0.1, delay=1.0)
        )
        
    populations[-1].record(["spikes"])
    
    print(f"Hardware simulation in progress: ({simulation_time} ms)...")
    sim.run(simulation_time)
    sim.end()
    
    print("Simulation completed with success.")

def organize_reports(dataset_name):
    report_dir = "reports"
    if not os.path.exists(report_dir):
        return
        
    subdirs = [os.path.join(report_dir, d) for d in os.listdir(report_dir) 
               if os.path.isdir(os.path.join(report_dir, d)) and d.startswith("20")]
               
    if not subdirs:
        return
        
    latest_report = max(subdirs, key=os.path.getmtime)
    folder_name = os.path.basename(latest_report)
    
    target_dir = os.path.join(report_dir, dataset_name)
    os.makedirs(target_dir, exist_ok=True)
    
    target_path = os.path.join(target_dir, folder_name)
    if not os.path.exists(target_path):
        shutil.copytree(latest_report, target_path)
        print(f"\n Energy report successfully sorted in : {target_path}/run_1/energy_report_1.rpt")

if DATASET == "nmnist":
    run_nmnist(PTH_FILE, SIMULATION_TIME)
    
elif DATASET == "cifar10":
    vgg4_layers = [1568, 50176, 25088, 12544, 4608, 10]
    run_vgg_topology("CIFAR10-DVS (SpikingVGG4)", vgg4_layers, SIMULATION_TIME)
    
elif DATASET == "dvs_gesture":
    vgg5_layers = [32768, 262144, 131072, 65536, 16384, 8192, 11]
    run_vgg_topology("DVS-Gesture (SpikingVGG5)", vgg5_layers, SIMULATION_TIME)

organize_reports(DATASET)